<a href="https://colab.research.google.com/github/DrikaGaribalde/DataScience/blob/main/Exercicios_laboratorio2Pos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Gelatina:
  def __init__(self, tam, cor, sabor):
      self.tam=tam
      self.cor=cor
      self.sabor=sabor

gel1 = Gelatina ("pequena", "vermelha", "morango")
gel2 = Gelatina ("media", "amarela", "abacaxi")
gel3 = Gelatina ("grande", "roxa", "uva")

print (gel1.tam, gel1.cor,gel1.sabor)
print (gel2.tam, gel2.cor,gel2.sabor)
print (gel3.tam, gel3.cor,gel3.sabor)



In [ ]:
class Produto:
  def __init__(self, cod, nome, quant):
    self.cod=cod
    self.nome=nome
    self.quant=quant  

codigo = input('Entre com o Codigo: ')
nome = input('Nome do produto: ')
quantidade = input('Qual a quantidade? ')
produt = Produto(codigo, nome, quantidade)
print ('\n')
print ('Dados de Saida:\n')
print ('Codigo: ' + produt.cod)
print ('Produto: ' + produt.nome)
print ('Quantidade: ' + produt.quant)

In [ ]:
# 

fibonacci = [0,1]

for i in range(2,10):
    fibonacci.append(fibonacci[i-2] + fibonacci[i-1])

print(fibonacci)

In [ ]:
import sys
sys.ps1= '>'

In [ ]:
for i in range(10):
 print(i)

In [ ]:
import re 
re.findall(r'\bf[a-z]','which foot or hand fell fastet')

['fo', 'fe', 'fa']

In [ ]:
teste1= re.sub(r'\bAMD', r'AuthenticAMD', 'AMD Turion(tm) 64 X2Mobile')
teste2= re.sub(r'(\b[a-z]+) \1', r'\1', 'Hoje esta um dia dia ensolarado')

print(teste1)
print(teste2)

In [ ]:
import math
def Sin(a):
# Calcula seno de angulo em graus
  ang = a*math.pi/180. # mesmo que radians()
  return math.sin(ang)
  #
Sin(30)
#Sin(60)

In [ ]:
import random
random.choice(['drika','felix','maria'])
#parei pagina 13 da apostila

Particionamento de Árvore de Decisão pagina 20 apostila

In [ ]:
from collections import Counter, defaultdict
from functools import partial
import math, random

def entropy(class_probabilities):
    """given a list of class probabilities, compute the entropy"""
    return sum(-p * math.log(p, 2) for p in class_probabilities if p)

def class_probabilities(labels):
    total_count = len(labels)
    return [count / total_count
            for count in Counter(labels).values()]

def data_entropy(labeled_data):
    labels = [label for _, label in labeled_data]
    probabilities = class_probabilities(labels)
    return entropy(probabilities)

def partition_entropy(subsets):
    """find the entropy from this partition of data into subsets"""
    total_count = sum(len(subset) for subset in subsets)

    return sum( data_entropy(subset) * len(subset) / total_count
                for subset in subsets )

def group_by(items, key_fn):
    """returns a defaultdict(list), where each input item
    is in the list whose key is key_fn(item)"""
    groups = defaultdict(list)
    for item in items:
        key = key_fn(item)
        groups[key].append(item)
    return groups

def partition_by(inputs, attribute):
    """returns a dict of inputs partitioned by the attribute
    each input is a pair (attribute_dict, label)"""
    return group_by(inputs, lambda x: x[0][attribute])

def partition_entropy_by(inputs,attribute):
    """computes the entropy corresponding to the given partition"""
    partitions = partition_by(inputs, attribute)
    return partition_entropy(partitions.values())

def classify(tree, input):
    """classify the input using the given decision tree"""

    # if this is a leaf node, return its value
    if tree in [True, False]:
        return tree

    # otherwise find the correct subtree
    attribute, subtree_dict = tree

    subtree_key = input.get(attribute)  # None if input is missing attribute

    if subtree_key not in subtree_dict: # if no subtree for key,
        subtree_key = None              # we'll use the None subtree

    subtree = subtree_dict[subtree_key] # choose the appropriate subtree
    return classify(subtree, input)     # and use it to classify the input

def build_tree_id3(inputs, split_candidates=None):

    # if this is our first pass,
    # all keys of the first input are split candidates
    if split_candidates is None:
        split_candidates = inputs[0][0].keys()

    # count Trues and Falses in the inputs
    num_inputs = len(inputs)
    num_trues = len([label for item, label in inputs if label])
    num_falses = num_inputs - num_trues

    if num_trues == 0:                  # if only Falses are left
        return False                    # return a "False" leaf

    if num_falses == 0:                 # if only Trues are left
        return True                     # return a "True" leaf

    if not split_candidates:            # if no split candidates left
        return num_trues >= num_falses  # return the majority leaf

    # otherwise, split on the best attribute
    best_attribute = min(split_candidates,
        key=partial(partition_entropy_by, inputs))

    partitions = partition_by(inputs, best_attribute)
    new_candidates = [a for a in split_candidates
                      if a != best_attribute]

    # recursively build the subtrees
    subtrees = { attribute : build_tree_id3(subset, new_candidates)
                 for attribute, subset in partitions.items() }

    subtrees[None] = num_trues > num_falses # default case

    return (best_attribute, subtrees)

def forest_classify(trees, input):
    votes = [classify(tree, input) for tree in trees]
    vote_counts = Counter(votes)
    return vote_counts.most_common(1)[0][0]


if __name__ == "__main__":

    inputs = [
        ({'level':'Senior','lang':'Java','tweets':'no','phd':'no'},   False),
        ({'level':'Senior','lang':'Java','tweets':'no','phd':'yes'},  False),
        ({'level':'Mid','lang':'Python','tweets':'no','phd':'no'},     True),
        ({'level':'Junior','lang':'Python','tweets':'no','phd':'no'},  True),
        ({'level':'Junior','lang':'R','tweets':'yes','phd':'no'},      True),
        ({'level':'Junior','lang':'R','tweets':'yes','phd':'yes'},    False),
        ({'level':'Mid','lang':'R','tweets':'yes','phd':'yes'},        True),
        ({'level':'Senior','lang':'Python','tweets':'no','phd':'no'}, False),
        ({'level':'Senior','lang':'R','tweets':'yes','phd':'no'},      True),
        ({'level':'Junior','lang':'Python','tweets':'yes','phd':'no'}, True),
        ({'level':'Senior','lang':'Python','tweets':'yes','phd':'yes'},True),
        ({'level':'Mid','lang':'Python','tweets':'no','phd':'yes'},    True),
        ({'level':'Mid','lang':'Java','tweets':'yes','phd':'no'},      True),
        ({'level':'Junior','lang':'Python','tweets':'no','phd':'yes'},False)
    ]

    for key in ['level','lang','tweets','phd']:
        print(key, partition_entropy_by(inputs, key))
    print()

    senior_inputs = [(input, label)
                     for input, label in inputs if input["level"] == "Senior"]

    for key in ['lang', 'tweets', 'phd']:
        print(key, partition_entropy_by(senior_inputs, key))
    print()

    print("building the tree")
    tree = build_tree_id3(inputs)
    print(tree)

    print("Junior / Java / tweets / no phd", classify(tree,
        { "level" : "Junior",
          "lang" : "Java",
          "tweets" : "yes",
          "phd" : "no"} ))

    print("Junior / Java / tweets / phd", classify(tree,
        { "level" : "Junior",
                 "lang" : "Java",
                 "tweets" : "yes",
                 "phd" : "yes"} ))

    print("Intern", classify(tree, { "level" : "Intern" } ))
    print("Senior", classify(tree, { "level" : "Senior" } ))

Perceptrons pag 29 apostila
https://linguagensdeprogramacao.wordpress.com/2011/09/03/perceptron-usando-python/

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

# aplicativo para verificar se o ser vivo eh quadrupede ou bipede
# quadrupede = 1, bipede = -1
# cao = [-1,-1,1,1] | resposta = 1
# gato = [1,1,1,1] | resposta = 1
# cavalo = [1,1,-1,1] | resposta = 1
# homem = [-1,-1,-1,1] | resposta = -1

# pesos (sinapses)
w = [0,0,0,0]
# entradas
x = [[-1,-1,1,1],
     [1,1,1,1],
     [1,1,-1,1],
     [-1,-1,-1,1]]
# respostas esperadas
t = [1,1,1,-1]
# bias (ajuste fino)
b = 0
#saida
y = 0
# numero maximo de interacoes
max_int = 10
# taxa de aprendizado
taxa_aprendizado = 1
#soma
soma = 0
#theshold
threshold = 1
# nome do animal
animal = ""
# resposta = acerto ou falha
resposta = ""

# dicionario de dados
d = {'-1,-1,1,1' : 'cao','1,1,1,1' : 'gato','1,1,-1,1' : 'cavalo','-1,-1,-1,1' : 'homem' }

print("Treinando")

# funcao para converter listas em strings
def listToString(list):
    s = str(list).strip('[]')
    s = s.replace(' ', '')
    return s

# inicio do algoritmo
for k in range(1,max_int):
    acertos = 0    
    print("INTERACAO "+str(k)+"-------------------------")
    for i in range(0,len(x)):
        soma = 0

        # pega o nome do animal no dicionário
        if (listToString(x[i])) in d:
            animal = d[listToString(x[i])]  
        else:
            animal = ""

        # para calcular a saida do perceptron, cada entrada de x eh multiplicada
        # pelo seu peso w correspondente
        for j in range(0,len(x[i])):
            soma += x[i][j] * w[j]

        # a saida eh igual a adicao do bias com a soma anterior
        y_in = b + soma
        #print("y_in = ",str(y_in))

        # funcao de saida eh determinada pelo threshold
        if y_in > threshold:
            y = 1
        elif y_in >= -threshold and y_in <= threshold:
            y = 0
        else:
            y = -1        

        # atualiza os pesos caso a saida nao corresponda ao valor esperado
        if y == t[i]:
            acertos+=1
            resposta = "acerto"
        else:
            for j in range (0,len(w)):                
                w[j] = w[j] + (taxa_aprendizado * t[i] * x[i][j])
            b = b + taxa_aprendizado * t[i]
            resposta = "Falha - Peso atualizado"

        #imprime a resposta
        if y == 1:
            print(animal+" = quadrupede = "+resposta)
        elif y == 0:
            print(animal+" = padrao nao identificado = "+resposta)
        elif y == -1:
            print(animal+" = bipede = "+resposta)

    if acertos == len(x):
        print("Funcionalidade aprendida com "+str(k)+" interacoes")
        break;
    print("")
print("Finalizado")


https://linguagensdeprogramacao.wordpress.com/2011/09/09/rede-neural-perceptron-controle-de-portas-and-em-python/
Redes Neurais Feed-Forward pag 33

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

# perceptron2.py
# aplicativo para analise de portas OR
# by: Antonio Rodrigo dos Santos Silva

# falso = 0, verdadeiro = 1

# [0,0] | resposta = 0
# [0,1] | resposta = 1
# [1,0] | resposta = 1
# [1,1] | resposta = 1

# numero maximo de interacoes
max_int = 20

# threshold (limiar)
threshold = 0

# peso 0
w_0 = -threshold

# entrada 0
x_0 = 1

# entradas
x = [[x_0,0,0],
     [x_0,0,1],
     [x_0,1,0],
     [x_0,1,1]]

# quantos itens tem o vetor x (4)
tamanho_x = len(x)

# quantos itens estão em cada posicao do vetor x
qtde_itens_x = len(x[0])

# pesos (sinapses)
w = [w_0,0,0]

# quantos itens tem o vetor w (3)
tamanho_w = len(w)

# respostas desejadas
d = [0,1,1,1]

# taxa de aprendizado (n)
taxa_aprendizado = 0.5

#saida
y = 0

# resposta = acerto ou falha
resposta = ""

# soma
u = 0

#erro
e = 0

print("Treinando")

# inicio do algoritmo
for k in range(1,max_int):
    acertos = 0    
    e = 0
    print("INTERACAO "+str(k)+"-------------------------")
    for t in range(0,tamanho_x):        
        u = 0

        # para calcular a saida do perceptron, cada entrada de x eh multiplicada
        # pelo seu peso w correspondente
        for j in range(0,qtde_itens_x):
            u += x[t][j] * w[j]

        # funcao de saida
        if u > 0:
            y = 1       
        else:
            y = 0

        # atualiza os pesos caso a saida nao corresponda ao valor esperado        
        if y == d[t]:
            resposta = "acerto"
            acertos += 1
            e = 0            
        else:
            resposta = "erro"        
            # calculando o erro
            e = d[t] - y
            # atualizando os pesos            
            for j in range (0,tamanho_w):
                w[j] = w[j] + (taxa_aprendizado * e * x[t][j])        

        print(resposta + " >>> u = "+str(u)+ ", y = "+ str(y)+ ", e = "+str(e))

    if acertos == tamanho_x:
        print("\nFuncionalidade aprendida com "+str(k)+" interacoes")
        print("\nPesos encontrados =============== ")
        for j in range (0,tamanho_w):
            print(w[j])
        break;
    print("")

print("Finalizado")

https://www.devmedia.com.br/redes-neurais-artificiais-algoritmo-backpropagation/28559
Backpropagation pag 38

In [ ]:
import math
import random
import string

class NN:
  def __init__(self, NI, NH, NO):
    # number of nodes in layers
    self.ni = NI + 1 # +1 for bias
    self.nh = NH
    self.no = NO
    
    # initialize node-activations
    self.ai, self.ah, self.ao = [],[], []
    self.ai = [1.0]*self.ni
    self.ah = [1.0]*self.nh
    self.ao = [1.0]*self.no

    # create node weight matrices
    self.wi = makeMatrix (self.ni, self.nh)
    self.wo = makeMatrix (self.nh, self.no)
    # initialize node weights to random vals
    randomizeMatrix ( self.wi, -0.2, 0.2 )
    randomizeMatrix ( self.wo, -2.0, 2.0 )
    # create last change in weights matrices for momentum
    self.ci = makeMatrix (self.ni, self.nh)
    self.co = makeMatrix (self.nh, self.no)
    
  def runNN (self, inputs):
    if len(inputs) != self.ni-1:
      print ('incorrect number of inputs')
    
    for i in range(self.ni-1):
      self.ai[i] = inputs[i]
      
    for j in range(self.nh):
      sum = 0.0
      for i in range(self.ni):
        sum +=( self.ai[i] * self.wi[i][j] )
      self.ah[j] = sigmoid (sum)
    
    for k in range(self.no):
      sum = 0.0
      for j in range(self.nh):        
        sum +=( self.ah[j] * self.wo[j][k] )
      self.ao[k] = sigmoid (sum)
      
    return self.ao
      
      
  
  def backPropagate (self, targets, N, M):
    # http://www.youtube.com/watch?v=aVId8KMsdUU&feature=BFa&list=LLldMCkmXl4j9_v0HeKdNcRA
    
    # calc output deltas
    # we want to find the instantaneous rate of change of ( error with respect to weight from node j to node k)
    # output_delta is defined as an attribute of each ouput node. It is not the final rate we need.
    # To get the final rate we must multiply the delta by the activation of the hidden layer node in question.
    # This multiplication is done according to the chain rule as we are taking the derivative of the activation function
    # of the ouput node.
    # dE/dw[j][k] = (t[k] - ao[k]) * s'( SUM( w[j][k]*ah[j] ) ) * ah[j]
    output_deltas = [0.0] * self.no
    for k in range(self.no):
      error = targets[k] - self.ao[k]
      output_deltas[k] =  error * dsigmoid(self.ao[k]) 
   
    # update output weights
    for j in range(self.nh):
      for k in range(self.no):
        # output_deltas[k] * self.ah[j] is the full derivative of dError/dweight[j][k]
        change = output_deltas[k] * self.ah[j]
        self.wo[j][k] += N*change + M*self.co[j][k]
        self.co[j][k] = change

    # calc hidden deltas
    hidden_deltas = [0.0] * self.nh
    for j in range(self.nh):
      error = 0.0
      for k in range(self.no):
        error += output_deltas[k] * self.wo[j][k]
      hidden_deltas[j] = error * dsigmoid(self.ah[j])
    
    #update input weights
    for i in range (self.ni):
      for j in range (self.nh):
        change = hidden_deltas[j] * self.ai[i]
        #print 'activation',self.ai[i],'synapse',i,j,'change',change
        self.wi[i][j] += N*change + M*self.ci[i][j]
        self.ci[i][j] = change
        
    # calc combined error
    # 1/2 for differential convenience & **2 for modulus
    error = 0.0
    for k in range(len(targets)):
      error = 0.5 * (targets[k]-self.ao[k])**2
    return error
        
        
  def weights(self):
    print ('Input weights:')
    for i in range(self.ni):
      print (self.wi[i])
    print
    print ('Output weights:')
    for j in range(self.nh):
      print (self.wo[j])
    print ('')
  
  def test(self, patterns):
    for p in patterns:
      inputs = p[0]
      print ('Inputs:', p[0], '-->', self.runNN(inputs), '\tTarget', p[1])
  
  def train (self, patterns, max_iterations = 1000, N=0.5, M=0.1):
    for i in range(max_iterations):
      for p in patterns:
        inputs = p[0]
        targets = p[1]
        self.runNN(inputs)
        error = self.backPropagate(targets, N, M)
      if i % 50 == 0:
        print('Combined error', error)
    self.test(patterns)
    

def sigmoid (x):
  return math.tanh(x)
  
# the derivative of the sigmoid function in terms of output
# proof here: 
# http://www.math10.com/en/algebra/hyperbolic-functions/hyperbolic-functions.html
def dsigmoid (y):
  return 1 - y**2

def makeMatrix ( I, J, fill=0.0):
  m = []
  for i in range(I):
    m.append([fill]*J)
  return m
  
def randomizeMatrix ( matrix, a, b):
  for i in range ( len (matrix) ):
    for j in range ( len (matrix[0]) ):
      matrix[i][j] = random.uniform(a,b)

def main ():
  pat = [
      [[0,0], [1]],
      [[0,1], [1]],
      [[1,0], [1]],
      [[1,1], [0]]
  ]
  myNN = NN ( 2, 2, 1)
  myNN.train(pat)
  
  





if __name__ == "__main__":
    main()


Combined error 0.43296234904077785
Combined error 0.006930148155448655
Combined error 0.002733745800504791
Combined error 0.0017119622434847244
Combined error 0.0012243105911451666
Combined error 0.0009461921590740417
Combined error 0.0007652878724936123
Combined error 0.0006489472580290689
Combined error 0.0005656044079727934
Combined error 0.0004938957184567878
Combined error 0.00044330351994799463
Combined error 0.0003992235670259502
Combined error 0.0003640916690975259
Combined error 0.00033410559639732537
Combined error 0.0003089512689634384
Combined error 0.0002867245221550516
Combined error 0.0002682583486930911
Combined error 0.0002509032904297365
Combined error 0.00023680464321861874
Combined error 0.00022313041098955585
Inputs: [0, 0] --> [0.998701550769033] 	Target [1]
Inputs: [0, 1] --> [0.9898227904494803] 	Target [1]
Inputs: [1, 0] --> [0.9897945630270857] 	Target [1]
Inputs: [1, 1] --> [0.013184842721846555] 	Target [0]


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

data = [ ("big data", 100, 15), ("Hadoop", 95, 25), ("Python", 75, 50),
("R", 50, 40), ("machine learning", 80, 20), ("statistics", 20, 60),
("data science", 60, 70), ("analytics", 90, 3),
("team player", 85, 85), ("dynamic", 2, 90), ("synergies", 70, 0),
("actionable insights", 40, 30), ("think out of the box", 45, 10),
("self-starter", 30, 50), ("customer focus", 65, 15),
("thought leadership", 35, 35)]

def text_size(total):
  """equals 8 if total is 0, 28 if total is 200"""
  return 8 + total / 200 * 20

  for word, job_popularity, resume_popularity in data:
    plt.text(job_popularity, resume_popularity, word,
      ha='center', va='center',
      size=text_size(job_popularity + resume_popularity))
  plt.xlabel("Popularity on Job Postings")
  plt.ylabel("Popularity on Resumes")
  plt.axis([0, 100, 0, 100])
  plt.show()

In [ ]:
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!
